In [5]:
import numpy as np
import torch

AttributeError: partially initialized module 'torch' from 'c:\Users\jules\OneDrive\Desktop\Opti App\.venv\Lib\site-packages\torch\__init__.py' has no attribute 'distributed' (most likely due to a circular import)

# Creating tensors

In [2]:
# From Python list
x = torch.tensor([1, 2, 3, 4, 5])
# Zeros tensor
zeros_tensor = torch.zeros(3, 4)
# Convert numpy array
numpy_array = np.array([[1, 2], [3, 4]])
tensor_from_numpy = torch.from_numpy(numpy_array)

NameError: name 'torch' is not defined

# A simple derivative

In [83]:
x = torch.tensor(3.321336, requires_grad=True)
y = x**3
z = 5*y
w = z + x
w.backward()
print(x.grad)


tensor(166.4691)


# In place operation does not work

In [86]:
x = torch.tensor(1.5, requires_grad=True)
x += 1 # this is the in place version of x = x + 1
y = x**2
y.backward()

RuntimeError: a leaf Variable that requires grad is being used in an in-place operation.

# Non leaf variable

In [89]:
# does not work
x = torch.tensor(1.5, requires_grad=True)
x = x + 1
y = x**2
y.backward()
print(x.grad)

None


C:\Users\lengo\AppData\Local\Temp\ipykernel_20216\1570524611.py:6: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more informations. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\build\aten\src\ATen/core/TensorBody.h:494.)
  print(x.grad)


In [91]:
# works
x = torch.tensor(1.5, requires_grad=True)
xnew = x + 1
y = xnew**2
y.backward()


# Gradient accumulation

In [94]:
# does not work
x = torch.tensor(1., requires_grad=True)
y = x**2
y.backward(retain_graph=True) # otherwise outputs are freed
print(x.grad)
y.backward()
print(x.grad)


tensor(2.)
tensor(4.)


In [96]:
# works
x = torch.tensor(1., requires_grad=True)
y = x**2
y.backward(retain_graph=True) # otherwise outputs are freed
print(x.grad)
x.grad.zero_()
y.backward()
print(x.grad)


tensor(2.)
tensor(2.)


# No gradient tracking

In [99]:
# does not work
x = torch.tensor(1., requires_grad=True)
y = x**2
with torch.no_grad(): # no gradient tracking inside this
    z = y + 1
    z.backward()

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

# Small loop

In [101]:

x = torch.tensor(1., requires_grad=True)
for k in range(10):
    y = x**2
    y.backward()
    with torch.no_grad():
        x -= 0.1 * x.grad # use in place operation
        x.grad.zero_()


In [103]:
# Built in GD

In [105]:
from torch.optim import SGD
x = torch.tensor(1., requires_grad=True)
optimizer = SGD([x], lr=0.1) # needs a list of parameters
for k in range(10):
    y = x**2
    print(x)
    y.backward()
    optimizer.step()
    optimizer.zero_grad()

tensor(1., requires_grad=True)
tensor(0.8000, requires_grad=True)
tensor(0.6400, requires_grad=True)
tensor(0.5120, requires_grad=True)
tensor(0.4096, requires_grad=True)
tensor(0.3277, requires_grad=True)
tensor(0.2621, requires_grad=True)
tensor(0.2097, requires_grad=True)
tensor(0.1678, requires_grad=True)
tensor(0.1342, requires_grad=True)


In [107]:

import torch.nn as nn
class LinearRegression(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(1, 1) # model parameters
    def forward(self, x):
        return self.linear(x) # evaluates prediction


model = LinearRegression()
criterion = nn.MSELoss() # Mean Squared Error loss
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
optimizer.zero_grad()

x = torch.tensor([1.], requires_grad=False)
y = torch.tensor([1.], requires_grad=False)

n_epochs = 1000 # number of iterations
losses = [] # to store losses



for epoch in range(n_epochs):
# Forward pass
    predictions = model(x)
    loss = criterion(predictions, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    losses.append(loss.item())



# GPU acceleration

In [110]:
# check cuda is available
print(torch.cuda.is_available())
# check devices
print(torch.cuda.device_count())
print(torch.cuda.current_device())
print(torch.cuda.get_device_name(0))

True
1
0
NVIDIA RTX 1000 Ada Generation Laptop GPU
